In [ ]:
import pandas as pd
from glob import glob
import json
from datetime import datetime
import numpy as np
from itertools import product

# import cf-units aware xarray. needs to be in this order
from xarray import open_mfdataset, open_dataset
import pint
import xarray as xr
import pint_xarray
import cf_xarray as cfxr # not side-effect free
import cf_xarray.units
xr.set_options(keep_attrs=True)
import xesmf as xe
import xcdat # SO many monkey patches

#import cfunits # different packages ;-;
#import cf_units

# note: have to --force-reinstall dask sometimes

In [ ]:
starting_year = 1993
last_CDIAC_year = 2021
last_EI_year = 2024

earth_radius = 6371.009 # km, John Miller's value

In [ ]:
EI_years = list(range(starting_year, last_EI_year+1))
EI_extrap_years = list(range(last_CDIAC_year, last_EI_year+1)) 

In [ ]:
CDIAC_global_xlsx = 'inputs/CDIAC/global.1751_2021.xlsx'
CDIAC_national_xlsx = 'inputs/CDIAC/nation.1751_2021.xlsx'
EI_xlsx = 'inputs/EI-Stats-Review-ALL-data-2025.xlsx'
EDGAR_ncs = 'inputs/TOTALS_flx_nc_2024_GHG/*.nc'
#EDGAR_ncs = 'inputs/TOTALS_emi_nc_v80/*.nc'
USGS_cement_csvs = './inputs/USGS_cement/mcs????-cement.csv'

### CDIAC at AppState
1. Nation and global annual emissions files
https://rieee.appstate.edu/projects-programs/cdiac/  
https://rieee.appstate.edu/wp-content/uploads/2024/07/global.1751_2020.xlsx  
https://rieee.appstate.edu/wp-content/uploads/2024/07/nation.1751_2020.xlsx  

Based on `read_cdiac_nation_csv.pro`

In [ ]:
CDIAC_global = pd.read_excel(CDIAC_global_xlsx, sheet_name='Sheet1')
CDIAC_global = CDIAC_global[CDIAC_global['Year'] >= starting_year].set_index('Year')
CDIAC_col_renames = { # teragrams
   'Total carbon emissions from fossil fuel consumption and cement production (million metric tons of C)':'total (Tg C)',
   'Carbon emissions from solid fuel consumption':'solid_fuel (Tg C)',
   'Carbon emissions from liquid fuel consumption':'liquid_fuel (Tg C)',
   'Carbon emissions from gas fuel consumption':'gas_fuel (Tg C)',
   'Carbon emissions from cement production':'cement (Tg C)',
   'Carbon emissions from gas flaring':'flaring (Tg C)',
   'Per capita carbon emissions (metric tons of carbon; after 1949 only)':'per_capita (Mg C)', # megagrams
}
CDIAC_global.rename(CDIAC_col_renames, axis='columns',inplace=True)

gg_cols = ['total (Gg C)', 'solid_fuel (Gg C)', 'liquid_fuel (Gg C)', 'gas_fuel (Gg C)', 'cement (Gg C)', 'flaring (Gg C)']
tg_cols = ['total (Tg C)', 'solid_fuel (Tg C)', 'liquid_fuel (Tg C)', 'gas_fuel (Tg C)', 'cement (Tg C)', 'flaring (Tg C)']
CDIAC_global[gg_cols] = 1000 * CDIAC_global[tg_cols] # convert to gigagrams (match with national)

output_cols = ['total (Gg C)','gas_fuel (Gg C)','liquid_fuel (Gg C)','solid_fuel (Gg C)','flaring (Gg C)','cement (Gg C)']
CDIAC_global.to_csv('processed_inputs/CDIAC_global_2020.csv', columns=output_cols)

assert len(CDIAC_global) == last_CDIAC_year - starting_year + 1, f"Expected {last_CDIAC_year - starting_year + 1} years, got {len(CDIAC_global)}"
assert not CDIAC_global[output_cols].isna().any().any(), "NaN in CDIAC global data"
display(CDIAC_global)

In [ ]:
renaming_list = {
    'PLURINATIONAL STATE OF BOLIVIA':'BOLIVIA',
    'HONG KONG SPECIAL ADMINSTRATIVE REGION OF CHINA':'HONG KONG',
    'CHINA (MAINLAND)':'CHINA',
    'MYANMAR (FORMERLY BURMA)':'MYANMAR',
    'BRUNEI (DARUSSALAM)':'BRUNEI',
    'DEMOCRATIC REPUBLIC OF THE CONGO (FORMERLY ZAIRE)':'DEMOCRATIC REPUBLIC OF THE CONGO',
    'FALKLAND ISLANDS (MALVINAS)':'FALKLAND ISLANDS',
    'FRANCE (INCLUDING MONACO)':'FRANCE',
    'LAO PEOPLE S DEMOCRATIC REPUBLIC':'LAOS',
    'LIBYAN ARAB JAMAHIRIYAH':'LIBYA',
    'RUSSIAN FEDERATION':'RUSSIA',
    'SYRIAN ARAB REPUBLIC':'SYRIA',
    'VIET NAM':'VIETNAM',
    'YUGOSLAVIA (MONTENEGRO & SERBIA)':'YUGOSLAVIA',
    'YUGOSLAVIA (FORMER SOCIALIST FEDERAL REPUBLIC)': 'YUGOSLAVIA'
}

# following would be nice to change later, but will screw up list alphabetization...
# ...possibly others too
# REPUBLIC OF CAMEROON
# REPUBLIC OF MOLDOVA
# UNITED REPUBLIC OF TANZANIA

aggregating_list = {
    'ETHIOPIA':['ETHIOPIA', 'ERITREA'],
    'ISRAEL':['ISRAEL', 'OCCUPIED PALESTINIAN TERRITORY'],
    'INDONESIA':['INDONESIA', 'TIMOR-LESTE (FORMERLY EAST TIMOR)'],
    'CANADA':['CANADA', 'ST. PIERRE & MIQUELON'],
    'SPAIN':['SPAIN', 'GIBRALTAR', 'ANDORRA'],
    'VENEZUELA':['VENEZUELA', 'ARUBA'],
    'CHINA':['CHINA', 'MACAU SPECIAL ADMINSTRATIVE REGION OF CHINA'],
    'YUGOSLAVIA':['YUGOSLAVIA', 'MACEDONIA', 'CROATIA','BOSNIA & HERZEGOVINA','SLOVENIA','KOSOVO', 'SERBIA', 'MONTENEGRO'],
    #'YUGOSLAVIA2': ['SERBIA', 'MONTENEGRO'], # Yugoslavia split to Serbia + Montenegro in 2006. Gets seperate entry to preserve per-capita correctly
    # we actually don't care about per-capita
    'SOUTH AFRICA':['SOUTH AFRICA', 'LESOTHO'],
    'UNITED KINGDOM':['UNITED KINGDOM', 'ISLE OF MAN'],
    'ST. KITTS-NEVIS':['ST. KITTS-NEVIS', 'ANGUILLA'],
    'GERMANY':['GERMANY', 'LIECHTENSTEIN'], # TODO: Germany or Switzerland?
    'SUDAN':['REPUBLIC OF SUDAN', 'REPUBLIC OF SOUTH SUDAN'] # Sudan split to Sudan + South Sudan in 2011
}

deleting_list = [
    'CAYMAN ISLANDS','NIUE','MONTSERRAT','PALAU','BRITISH VIRGIN ISLANDS','ANTARCTIC FISHERIES','SAINT HELENA',
    'WALLIS AND FUTUNA ISLANDS','MARSHALL ISLANDS','FEDERATED STATES OF MICRONESIA','TURKS AND CAICOS ISLANDS',
    'BONAIRE, SAINT EUSTATIUS, AND SABA','CURACAO','NETHERLAND ANTILLES','SAINT MARTIN (DUTCH PORTION)','TUVALU','MAYOTTE',
]

CDIAC_col_renames = {
    'Total CO2 emissions from fossil-fuels and cement production (thousand metric tons of C)':'total (Gg C)',
    'Emissions from solid fuel consumption':'solid_fuel (Gg C)',
    'Emissions from liquid fuel consumption':'liquid_fuel (Gg C)',
    'Emissions from gas fuel consumption':'gas_fuel (Gg C)',
    'Emissions from cement production':'cement (Gg C)',
    'Emissions from gas flaring':'flaring (Gg C)',
    'Per capita CO2 emissions (metric tons of carbon)':'per_capita',
    'Emissions from bunker fuels (not included in the totals)':'bunker (Gg C)'
}

# in 2013 version, (BONAIRE, SAINT EUSTATIUS, AND SABA), CURACAO, LICHTENSTEIN, NETHERLAND ANTILLES,SAINT MARTIN (DUTCH PORTION) were added (they do not have full records from 1992 - 2013, which makes
# implementation in ff_country_new20xx.pro difficult.)
# in 2014 version, delete TUVALU
# 
# Liechtenstein now appears fine in CDIAC data that goes through 2017. So merge with Switzerland for now instead of deleting

# NOTE:  The following countries do have GISS codes -- although they may not have full records from 1992 onwards as mentioned above
# Liechtenstein, Netherlands Antilles, Turks and Caicos, Tuvalu

def aggregate_countries(data, little_names, big_name) :
    aggregated = data.xs(key=little_names[0], level="Nation").add(data.xs(key=little_names[1], level="Nation"), fill_value=0)
    for little_name in little_names[2:]:
        aggregated = aggregated.add(data.xs(key=little_name, level="Nation"), fill_value=0)
    aggregated['per_capita'] = data.xs(key=little_names[0], level="Nation")['per_capita'] # intensive, just assume ~first little
    aggregated = pd.concat({big_name: aggregated}, names=['Nation']) # add nation index back in
    return pd.concat([data.drop(little_names, level='Nation'), aggregated]) # remove originals


In [ ]:
CDIAC_national = pd.read_excel(CDIAC_national_xlsx, sheet_name='Sheet1').set_index(['Nation', 'Year'])#.fillna(0)

# give countries+cols more standard names
CDIAC_national.rename(renaming_list, level=0, inplace=True)
CDIAC_national.rename(CDIAC_col_renames, axis='columns', inplace=True)

# merge small countries (e.g. Eritrea) onto neighboring big countries. Also glue Yugoslavia and Sudan back together.
for big, littles in aggregating_list.items():
    CDIAC_national = aggregate_countries(CDIAC_national, littles, big)
CDIAC_national.rename({'YUGOSLAVIA2':'YUGOSLAVIA'}, level=0, inplace=True)

# delete even smaller countries
CDIAC_national.drop(deleting_list, level=0, inplace=True)

# only keep data after year countries stabilize, fix indices
CDIAC_national.reset_index(inplace=True) 
CDIAC_national['Nation'] = CDIAC_national['Nation'].str.title()
CDIAC_national = CDIAC_national[CDIAC_national['Year'] >= starting_year] 
CDIAC_national.set_index(['Nation', 'Year'], inplace=True)

# French departments are missing values for 2011-2014, so interpolate
for department, year in product(['French Guiana','Guadeloupe','Martinique','Reunion'], [2011, 2012, 2013, 2014]):
    CDIAC_national.loc[(department,year), :] = np.nan 
CDIAC_national.sort_index(axis='index', inplace=True) # needs to be in correct order for interpolation

CDIAC_countries = CDIAC_national.index.get_level_values('Nation').unique().to_series()

# need to ONLY interpolate in a given nation (whoops...)
for nation in CDIAC_national.index.get_level_values('Nation').unique(): # df -> series, fill na, -> df
    CDIAC_national.loc[[nation]] = pd.concat({nation:  CDIAC_national.loc[nation].interpolate()}, names=['Nation'])


# output. Interpolate missing values, then fill extrapolated missing values with zero
output_cols = ['total (Gg C)','gas_fuel (Gg C)','liquid_fuel (Gg C)','solid_fuel (Gg C)','flaring (Gg C)','cement (Gg C)']
CDIAC_national_filled = CDIAC_national.fillna(0)
CDIAC_national_filled.to_csv('processed_inputs/CDIAC_national_2020.csv', columns=output_cols)
CDIAC_countries.to_csv('processed_inputs/CDIAC_countries.csv', columns=[], header=False)

# --- validation ---
n_nations = CDIAC_national.index.get_level_values('Nation').nunique()
n_years = CDIAC_national.index.get_level_values('Year').nunique()
assert n_years == last_CDIAC_year - starting_year + 1, f"Expected {last_CDIAC_year - starting_year + 1} years, got {n_years}"

# every country should have exactly n_years rows (no gaps)
rows_per_nation = CDIAC_national_filled.groupby('Nation').size()
incomplete = rows_per_nation[rows_per_nation != n_years]
assert incomplete.empty, f"Countries with incomplete year coverage:\n{incomplete}"

# year axis should be contiguous 1993..2021 with no duplicates per country
expected_years = list(range(starting_year, last_CDIAC_year + 1))
for nation in CDIAC_national_filled.index.get_level_values('Nation').unique()[:5]:  # spot-check first 5
    nation_years = sorted(CDIAC_national_filled.loc[nation].index.get_level_values('Year'))
    assert nation_years == expected_years, f"{nation}: years are {nation_years[:5]}..., expected {expected_years[:5]}..."
assert not CDIAC_national_filled.index.duplicated().any(), "Duplicate (Nation, Year) rows in CDIAC national"

# sector columns should sum to total (gas + liquid + solid + cement + flaring = total)
sector_cols = ['gas_fuel (Gg C)', 'liquid_fuel (Gg C)', 'solid_fuel (Gg C)', 'flaring (Gg C)', 'cement (Gg C)']
sector_sum = CDIAC_national_filled[sector_cols].sum(axis=1)
reported_total = CDIAC_national_filled['total (Gg C)']
sector_mismatch = (sector_sum - reported_total).abs()
worst_mismatch = sector_mismatch.max()
if worst_mismatch > 1.0:  # allow 1 Gg rounding
    print(f"  WARNING: sector sum != reported total, max mismatch = {worst_mismatch:.1f} Gg C")
else:
    print(f"  Sector sum check OK (max mismatch {worst_mismatch:.2f} Gg C)")

# negative emissions warning — Estonia and Oman have negative liquid_fuel values
# in recent years (likely net fuel exports reported as negative consumption in CDIAC)
for col in output_cols:
    neg_rows = CDIAC_national_filled[CDIAC_national_filled[col] < 0]
    if len(neg_rows) > 0:
        nations = neg_rows.index.get_level_values('Nation').unique().tolist()
        print(f"  WARNING: {len(neg_rows)} rows with negative {col}: {nations}")

# global total should exceed sum of country totals (difference = bunker fuels)
country_sums = CDIAC_national_filled.groupby('Year')['total (Gg C)'].sum()
global_totals = CDIAC_global['total (Gg C)']
bunker_frac = (global_totals - country_sums) / global_totals
assert (global_totals >= country_sums).all(), "Country totals exceed global total (bunker fuels would be negative)"
if (bunker_frac > 0.10).any():
    print(f"  WARNING: bunker fraction exceeds 10% in some years: max = {bunker_frac.max():.1%}")

print(f"  {n_nations} nations, {n_years} years, bunker fraction {bunker_frac.min():.1%}--{bunker_frac.max():.1%}")
display(CDIAC_national)

### BP/EI data for extrapolation

https://www.energyinst.org/statistical-review
https://www.energyinst.org/__data/assets/excel_doc/0020/1540550/EI-Stats-Review-All-Data.xlsx

—> use oil, coal, and gas all in energy units — exajoules.

In [ ]:
EI_renaming = {
    'China Hong Kong SAR':'Hong Kong',
    'Iran':'Islamic Republic Of Iran',
    'Republic of Ireland':'Ireland',
    'South Korea':'Republic Of Korea',
    'US':'United States Of America',
    'Trinidad & Tobago':'Trinidad And Tobago',
    'Italy':'Italy (Including San Marino)',
    'Russian Federation':'Russia',
    'Brunei (Darussalam)':'Brunei',
    'Democratic Republic Of The Congo (Formerly Zaire)' : 'Democratic Republic Of The Congo',
    'Falkland Islands (Malvinas)': 'Falkland Islands',
    'North Macedonia': 'Macedonia'
}

#### Global (for extrapolation)

In [ ]:
def _read_ei_global(sheet, skipfooter):
    """Read EI global row, keeping only integer-year columns."""
    df = pd.read_excel(EI_xlsx, sheet_name=sheet, header=2, index_col=0, skipfooter=skipfooter)
    assert 'Total World' in df.index, f"'Total World' not found in sheet '{sheet}' -- check skipfooter"
    row = df.loc['Total World']
    # keep only columns that parse as integer years (drop '2024.1', '2014-24', etc.)
    year_cols = [c for c in row.index if isinstance(c, (int, float)) and float(c) == int(c)]
    return row[year_cols].rename(lambda c: int(c))

global_oil     = _read_ei_global('Primary Energy Cons (old meth)', 10)
global_gas     = _read_ei_global('Gas Consumption - EJ', 14)
global_coal    = _read_ei_global('Coal Consumption - EJ', 13)
global_flaring = _read_ei_global('CO2 from Flaring', 13)

global_EI = pd.concat([global_oil,global_gas,global_coal, global_flaring], axis=1, keys=['oil', 'gas', 'coal', 'flaring']).stack().unstack(0)
display(global_EI)

In [ ]:
(global_oil.pct_change() + 1).dropna().to_csv('processed_inputs/EI_frac_changes_2020-2024_global_oil.csv', index=False)
(global_gas.pct_change() + 1).dropna().to_csv('processed_inputs/EI_frac_changes_2020-2024_global_gas.csv', index=False)
(global_coal.pct_change() + 1).dropna().to_csv('processed_inputs/EI_frac_changes_2020-2024_global_coal.csv', index=False)

#### Flaring

In [ ]:
# flaring has a different country set, so treat separately
EI_flaring = pd.read_excel(EI_xlsx, sheet_name='CO2 from Flaring', index_col=0, header=2, skipfooter=13).dropna(axis='index', how='all').dropna(axis='columns', how='all')
EI_flaring.index.names = ['Nation']
EI_flaring.rename(EI_renaming, level='Nation', inplace=True)
EI_flaring.reset_index(inplace=True)
EI_flaring = EI_flaring[~EI_flaring['Nation'].str.startswith('Total') & ~EI_flaring['Nation'].str.lower().str.startswith('of which')].set_index('Nation') # remove totals and subtotals

display(EI_flaring.head())

In [ ]:
with open('EI_2024_flaring_regions.json') as f:
    EI_flaring_regions = json.load(f)

# No Macedonia, Slovenia, Croatia, or USSR to worry about

# take region e.g. Other Europe, copy values to all countries inside it, and remove
for EI_label, CDIAC_countries in EI_flaring_regions.items():
    for CDIAC_country in CDIAC_countries:
        EI_flaring.loc[CDIAC_country, :] = EI_flaring.loc[EI_label, :]
EI_flaring.drop(EI_flaring_regions.keys(), inplace=True)

EI_flaring.reset_index(inplace=True)
EI_flaring['Nation'] = EI_flaring['Nation'].str.title()
EI_flaring.set_index('Nation',inplace=True)
EI_flaring.sort_index(inplace=True)

display(EI_flaring)

#### Oil, gas, and coal

In [ ]:
# different footer amounts to remove comments and world totals
EI_oil = pd.read_excel(EI_xlsx, sheet_name='Primary Energy Cons (old meth)', header=2, index_col=0, skipfooter=10).dropna(axis='index', how='all').dropna(axis='columns', how='all')
EI_gas = pd.read_excel(EI_xlsx, sheet_name='Gas Consumption - EJ', header=2, index_col=0, skipfooter=14).dropna(axis='index', how='all').dropna(axis='columns', how='all')
EI_coal = pd.read_excel(EI_xlsx, sheet_name='Coal Consumption - EJ', header=2, index_col=0, skipfooter=13).dropna(axis='index', how='all').dropna(axis='columns', how='all')

# validate that skipfooter didn't eat the data rows
for label, df in [('oil', EI_oil), ('gas', EI_gas), ('coal', EI_coal)]:
    assert 'Total World' in df.index, f"'Total World' missing from EI {label} -- check skipfooter"

In [ ]:
EI_fuels = pd.concat({'oil': EI_oil, 'coal': EI_coal, 'gas': EI_gas}, names=['Fuel_type'])
EI_fuels.index.names = ['Fuel_type', 'Nation']

EI_fuels.rename(EI_renaming, level='Nation', inplace=True)
#EI_fuels = EI_fuels[EI_years] # only keep EI years (e.g. 1993 - 2023)
EI_fuels = EI_fuels[~EI_fuels.index.get_level_values('Nation').str.startswith('Total') & ~EI_fuels.index.get_level_values('Nation').str.lower().str.startswith('of which')] # remove totals and subtotals
EI_fuels.reset_index('Nation', inplace=True)

# we put Macedonia, Slovenia, and Croatia in Yugoslavia, which is in 'other Europe', so combine there
yugoslavia_extras = EI_fuels[EI_fuels['Nation'] == 'Macedonia'].add(EI_fuels[EI_fuels['Nation'] == 'Slovenia'].add(EI_fuels[EI_fuels['Nation'] == 'Croatia']))
yugoslavia_extras['Nation'] = ''
EI_fuels[EI_fuels['Nation'] == 'Other Europe'] = EI_fuels[EI_fuels['Nation'] == 'Other Europe'].add(yugoslavia_extras)
EI_fuels = EI_fuels.reset_index('Fuel_type').set_index('Nation')
EI_fuels.drop(['Macedonia', 'Slovenia', 'Croatia', 'USSR'], axis='index', inplace=True)# also remove the USSR

EI_fuels = EI_fuels.reset_index().set_index(['Nation', 'Fuel_type'])
EI_fuels.sort_index(inplace=True)
EI_fuels

In [ ]:
# NOTE: Afghanistan moved to 'Other Asia Pacific', in accordance with EI Definitions tab
with open('EI_2024_fuel_regions.json') as f:
    EI_fuel_regions = json.load(f)

In [ ]:
# checks that countries are appropriately represented in CDIAC, EI, and regions
all_extras = []
for nations in EI_fuel_regions.values():
    all_extras += nations
    
Extra_countries = set(all_extras)
CDIAC_countries_set = set(CDIAC_national.index.get_level_values(0).str.title().unique())
EI_countries = set(EI_fuels.index.get_level_values(0).str.title().unique())

# should both be empty sets
missing_countries = (CDIAC_countries_set - EI_countries) - Extra_countries
duplicate_countries = Extra_countries - (CDIAC_countries_set - EI_countries)
assert not missing_countries, f"CDIAC countries missing from EI + regions: {missing_countries}"
assert not duplicate_countries, f"Region countries duplicating EI entries: {duplicate_countries}"
print(f"  Country check passed: {len(CDIAC_countries_set)} CDIAC, {len(EI_countries)} EI, {len(Extra_countries)} from regions")

In [ ]:
# take region e.g. Middle Africa, copy values to all countries inside it, and remove
for EI_label, CDIAC_nations in EI_fuel_regions.items():
    for CDIAC_country in CDIAC_nations:
        dupe = pd.concat({CDIAC_country: EI_fuels.loc[EI_label]}, names=['Nation'])
        EI_fuels = pd.concat([EI_fuels, dupe])

EI_fuels.sort_index(inplace=True)
EI_fuels.drop(EI_fuel_regions.keys(), inplace=True)

EI_fuels.reset_index(inplace=True)
EI_fuels['Nation'] = EI_fuels['Nation'].str.title()
EI_fuels.set_index(['Nation', 'Fuel_type'], inplace=True)
EI_fuels

#### Combine Flaring with others

In [ ]:
EI_combined = EI_fuels.reset_index().set_index(['Fuel_type', 'Nation']).sort_index()
x = pd.concat([EI_flaring], keys=['flaring'], names=['Fuel_type'])
EI_combined = pd.concat([EI_combined, x]).reset_index()
EI_combined = EI_combined.set_index(['Nation', 'Fuel_type']).sort_index()
EI_combined.loc['Spain', 'flaring'] += EI_combined.loc['Gibraltar', 'flaring'] # merge Gibraltar (only flaring) onto Spain 
EI_combined.drop('Gibraltar', axis='index', level='Nation', inplace=True)

# every CDIAC country should have all 4 fuel types (oil, gas, coal, flaring)
expected_fuels = {'coal', 'flaring', 'gas', 'oil'}
nations_in_combined = EI_combined.index.get_level_values('Nation').str.title().unique()
for nation in CDIAC_countries:
    if nation not in nations_in_combined:
        print(f"  WARNING: CDIAC country '{nation}' missing entirely from EI combined")
    else:
        fuels_present = set(EI_combined.loc[nation].index.get_level_values('Fuel_type'))
        missing_fuels = expected_fuels - fuels_present
        assert not missing_fuels, f"Country '{nation}' missing fuel types: {missing_fuels}"

# EI per-country totals should approximately match the global 'Total World' row
# (won't be exact due to region-assignment and rounding, but should be close)
for fuel, global_series in [('oil', global_oil), ('gas', global_gas), ('coal', global_coal)]:
    national_sum = EI_combined.xs(fuel, level='Fuel_type')[EI_years].sum(axis=0)
    global_vals = global_series[EI_years]
    rel_err = ((national_sum - global_vals) / global_vals).abs()
    worst_year = rel_err.idxmax()
    if rel_err.max() > 0.05:
        print(f"  WARNING: EI {fuel} national sum vs global differs by {rel_err.max():.1%} in {worst_year}")
    else:
        print(f"  EI {fuel} national vs global OK (max {rel_err.max():.1%} in {worst_year})")

EI_combined_ranged = EI_combined[EI_years]
EI_csv = EI_combined_ranged.stack().unstack(level='Fuel_type').rename(columns={'coal':'coal (EJ)', 'flaring':'flaring (TG CO2)', 'oil':'oil (EJ)', 'gas':'gas (EJ)'}) # transpose fuel_type to columns

EI_csv.to_csv('processed_inputs/EI_national_2024.csv')

#### Ratios for extrapolation

In [ ]:
fractional_changes = EI_combined[[last_CDIAC_year-1] + EI_extrap_years].stack().unstack(level=1).groupby(level='Nation').pct_change().drop(labels=last_CDIAC_year-1, level=1).replace([np.inf, -np.inf], np.nan) + 1

global_fractional_changes = global_EI.T.pct_change().T[EI_extrap_years].T + 1

for nation in fractional_changes.index.get_level_values('Nation').unique(): # df -> series, fill na, -> df
    fractional_changes.loc[[nation]] = pd.concat({nation:  fractional_changes.loc[nation].fillna(global_fractional_changes)}, names=['Nation'])

fractional_changes['gas'].unstack().drop(['Ussr']).to_csv('processed_inputs/EI_frac_changes_2020-2024_gas.csv')
fractional_changes['coal'].unstack().drop(['Ussr']).to_csv('processed_inputs/EI_frac_changes_2020-2024_coal.csv')
fractional_changes['oil'].unstack().drop(['Ussr']).to_csv('processed_inputs/EI_frac_changes_2020-2024_oil.csv')

# validate: each fuel should have one row per CDIAC nation, with no remaining NaNs
n_expected = CDIAC_national.index.get_level_values('Nation').nunique()
for fuel in ['gas', 'coal', 'oil', 'flaring']:
    fc = fractional_changes[fuel].unstack().drop(['Ussr'], errors='ignore')
    assert len(fc) == n_expected, f"EI {fuel} ratios: {len(fc)} nations, expected {n_expected}"
    assert fc.notna().all().all(), f"NaN in EI {fuel} fractional changes after fill"

# warn about extreme year-over-year ratios (outside [0.5, 2.0] means >2x or <0.5x change)
for fuel in ['gas', 'coal', 'oil', 'flaring']:
    fc = fractional_changes[fuel].unstack().drop(['Ussr'], errors='ignore')
    extreme = (fc < 0.5) | (fc > 2.0)
    n_extreme = extreme.sum().sum()
    if n_extreme > 0:
        worst_nations = fc[extreme.any(axis=1)].index.tolist()
        print(f"  WARNING: {n_extreme} extreme EI {fuel} ratios (outside [0.5, 2.0]) in: {worst_nations[:5]}{'...' if len(worst_nations) > 5 else ''}")

display(fractional_changes)

### USGS Cement
https://www.usgs.gov/centers/national-minerals-information-center/cement-statistics-and-information

In [ ]:
USGS_renames = {
    'United States (includes Puerto Rico)':'United States of America',
    'Korea Republic of':'Republic of Korea'
}

# these CSVs are constructed by manually reading the PDF
cement_datas = []
for cement_datafile in sorted(glob(USGS_cement_csvs)):
    cement_data = pd.read_csv(cement_datafile)
    cement_data.columns = cement_data.columns.str.replace(' (Gg)', '')
    cement_data = pd.wide_to_long(cement_data, ['Cement', 'Clinker'], i='Nation', j='Year', sep=' ')
    cement_datas.append(cement_data)
    
total_cement = pd.concat(cement_datas).sort_index().rename(USGS_renames, axis='index')
total_cement = total_cement.reset_index().drop_duplicates(subset=['Nation', 'Year'], keep='last').set_index(['Nation', 'Year']).sort_index()

total_cement.to_csv('./processed_inputs/USGS_cement_2026.csv')

In [ ]:
cement_ratios = total_cement.drop(columns='Clinker').drop(labels='World total (rounded)').stack().unstack(level='Year')

# check for zero base year before dividing — countries with zero 2020 cement would produce inf
zero_base = cement_ratios[cement_ratios[2020] == 0]
if len(zero_base) > 0:
    print(f"  WARNING: {len(zero_base)} countries have zero cement production in base year 2020 (will be dropped by dropna):")
    print(f"    {zero_base.index.get_level_values('Nation').unique().tolist()}")

cement_ratios = cement_ratios.div(cement_ratios[2020], axis=0).dropna(how='all')
cement_other_nations = cement_ratios.xs('Other countries (rounded)').reset_index()
cement_ratios = cement_ratios.reset_index().drop(columns=['level_1'])

new_rows = []
for CDIAC_country in CDIAC_countries:
    if not (CDIAC_country in list(cement_ratios['Nation'])):
        temp = cement_other_nations.copy()
        temp['Nation'] = CDIAC_country
        new_rows.append(temp)

cement_ratios = pd.concat([cement_ratios, *new_rows]).set_index('Nation').sort_index()
cement_ratios = cement_ratios.drop('Other countries (rounded)').drop(columns=[c for c in ['index', 2018, 2019] if c in cement_ratios.columns])
cement_ratios = cement_ratios.stack('Year').to_frame()
cement_ratios.columns = ['cement']

# cement ratios should be positive (ratio to 2020 baseline)
assert (cement_ratios['cement'] > 0).all(), f"Non-positive cement ratios found:\n{cement_ratios[cement_ratios['cement'] <= 0]}"
# warn about extreme cement ratios
extreme_cement = cement_ratios[(cement_ratios['cement'] < 0.1) | (cement_ratios['cement'] > 10)]
if len(extreme_cement) > 0:
    print(f"  WARNING: {len(extreme_cement)} extreme cement ratios (outside [0.1, 10x of 2020]):")
    print(extreme_cement.head(10).to_string())

cement_ratios.to_csv('processed_inputs/USGS_cement_ratios_2020-2026.csv')
display(cement_ratios)

### EDGAR emissions for spatial patterning
https://edgar.jrc.ec.europa.eu/gallery?release=v2024ghg&substance=CO2&sector=TOTALS  
https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2024_GHG/CO2/TOTALS/TOTALS_flx_nc.zip

EDGAR v8 is quite different from EDGAR_GHG_2024, so needs different processing

In [ ]:
print(sorted(glob(EDGAR_ncs)))

In [ ]:
grid_1x1 = xe.util.grid_global(1, 1, cf=True)
grid_1x1_areas = xe.util.cell_area(grid_1x1, earth_radius=earth_radius)

grid_01x01 = xe.util.grid_global(.1, .1, cf=True)
grid_01x01_areas = xe.util.cell_area(grid_01x01, earth_radius=earth_radius)

def add_id(ds):
    ds.coords['year'] = int(ds.variables['fluxes'].attrs['year'])
    return ds

def seconds_in_year(year): # since it changes per year. NOTE: doesn't handle leap seconds correctly (unix just smears them). Probably doesn't matter.
    return (datetime(year+1, 1, 1, 0, 0, 0).timestamp() - datetime(year, 1, 1, 0, 0, 0).timestamp()) 

# note: FAKE files duplicate 2023 for 2024 and 2025
EDGAR_flux_all_years = open_mfdataset(sorted(glob(EDGAR_ncs)), preprocess=add_id, combine='nested', concat_dim='year')
EDGAR_flux_all_years['fluxes'] = EDGAR_flux_all_years['fluxes'].pint.quantify(EDGAR_flux_all_years['fluxes'].attrs['units']) #  pint.Unit('Mg')
# TODO: move secs_per_year to here

starting_index = list(EDGAR_flux_all_years['year']).index(starting_year)

# Extend EDGAR data to cover through 2025 by duplicating the last available year (if FAKE files not present)
last_year = int(EDGAR_flux_all_years['year'].values[-1])
target_year = last_EI_year + 1  # = 2025; must match yr3 in ff_country_2026.pro
if last_year < target_year:
    extras = []
    for yr in range(last_year + 1, target_year + 1):
        extra = EDGAR_flux_all_years['fluxes'][-1:, :, :].copy()
        extra['year'] = [yr]
        extras.append(extra)
    extended_fluxes = xr.concat([EDGAR_flux_all_years['fluxes']] + extras, dim='year')
else:
    extended_fluxes = EDGAR_flux_all_years['fluxes']

EDGAR_flux = xr.Dataset()
EDGAR_flux['fluxes'] = extended_fluxes[starting_index:, : :]
EDGAR_flux['01x01 areas'] = xr.DataArray(grid_01x01_areas.to_numpy(), dims=['lat','lon']).pint.quantify('km^2') # force floating point lat/lon to line up
EDGAR_flux['seconds_per_year'] = xr.DataArray(np.array([seconds_in_year(int(year)) for year in EDGAR_flux['year']]) * cf_xarray.units.units('s'), dims='year')
EDGAR_flux['emissions'] = EDGAR_flux['fluxes'] * EDGAR_flux['01x01 areas'] * EDGAR_flux['seconds_per_year']
#EDGAR_flux['flux'] = EDGAR_flux['fluxes'] #EDGAR_flux['emissions'] / EDGAR_flux['01x01 areas'] / EDGAR_flux['seconds_per_year']
EDGAR_flux.attrs = EDGAR_flux_all_years.attrs

EDGAR_flux['fluxes'][0,:,:].plot(robust=True)

display(EDGAR_flux)

In [ ]:
# regrid to 1x1. takes 50 seconds.
# strip units for regridder, then re-add them
#regridder = xe.Regridder(EDGAR_flux['emissions'][0,:,:].pint.dequantify(), grid_1x1, "conservative", periodic=True) # expensive part

In [ ]:
num_years = len(EDGAR_flux['year'])

# do it in a loop because constructing the regridder is the expensive part, so don't want to make giant one
aggregates = []
for year_index in range(num_years):
    # for unknown reasons, regridding (on flux) with ESMF produces different results than manually aggregating (on emissions)
    #aggregate = regridder(EDGAR_flux['flux'][year_index,:,:].pint.to('g / m^2 / s').pint.dequantify()).pint.quantify('g / m^2 / s') 
    ems = EDGAR_flux['emissions'][year_index,:,:].pint.to('Mg').to_numpy()
    aggregate = xr.DataArray(ems.reshape(1800, 360, 10).sum(axis=2).T.reshape(360, 180, 10).sum(axis=2).T, dims=['lat', 'lon']).pint.quantify('Mg')
    aggregate.attrs = aggregate.attrs | EDGAR_flux['emissions'][year_index,:,:].attrs # just glue the attributes back on. probably fine.
    aggregates.append(aggregate)
    
# CF convention stuffs
EDGAR_aggregated_flux = xr.Dataset()
EDGAR_aggregated_flux['emissions'] = xr.concat(aggregates, dim='year')
EDGAR_aggregated_flux['cell area'] = xr.DataArray(grid_1x1_areas.to_numpy(), dims=['lat','lon']).pint.quantify('km^2') # force lat/lon to line up
EDGAR_aggregated_flux['seconds_per_year'] = EDGAR_flux['seconds_per_year']

EDGAR_aggregated_flux['flux'] = EDGAR_aggregated_flux['emissions'] / EDGAR_aggregated_flux['cell area'] / EDGAR_aggregated_flux['seconds_per_year']
EDGAR_aggregated_flux['mass_tendency'] = (EDGAR_aggregated_flux['flux'] * EDGAR_aggregated_flux['cell area'] * EDGAR_aggregated_flux['seconds_per_year']).pint.to('Pg')


EDGAR_aggregated_flux.attrs = EDGAR_flux.attrs | {'earth_radius': earth_radius}
EDGAR_aggregated_flux['lon_bounds'] = grid_1x1['lon_bounds']
EDGAR_aggregated_flux['lat_bounds'] = grid_1x1['lat_bounds']
del EDGAR_aggregated_flux['latitude_longitude']
del EDGAR_aggregated_flux['flux'].attrs['year']
del EDGAR_aggregated_flux['flux'].attrs['global_total'] # need to sum
EDGAR_aggregated_flux['mass_tendency'].attrs['longname'] = "Mass of carbon emitted into atmosphere each year by burning fossil fuels, per cell"
EDGAR_aggregated_flux['flux'].attrs['longname'] = "Flux of carbon emitted into atmosphere by burning fossil fuels"

EDGAR_aggregated_flux['normalized_flux'] = EDGAR_aggregated_flux['flux'] / EDGAR_aggregated_flux['flux'].sum(dim=['lat','lon'])
EDGAR_aggregated_flux['normalized_mass_tendency'] = EDGAR_aggregated_flux['mass_tendency'] / EDGAR_aggregated_flux['mass_tendency'].sum(dim=['lat','lon'])

expected_years = target_year - starting_year + 1
actual_years = len(EDGAR_aggregated_flux['year'])
assert actual_years == expected_years, f"Expected {expected_years} years in EDGAR, got {actual_years}"

# verify EDGAR year labels are exactly [1993, 1994, ..., target_year]
edgar_year_labels = [int(y) for y in EDGAR_flux['year'].values]
expected_year_labels = list(range(starting_year, target_year + 1))
assert edgar_year_labels == expected_year_labels, f"EDGAR year labels {edgar_year_labels[:5]}...{edgar_year_labels[-3:]} != expected {expected_year_labels[:5]}...{expected_year_labels[-3:]}"

# non-negative fluxes — fossil fuel emissions should never be negative
for yi in range(num_years):
    ems = EDGAR_aggregated_flux['emissions'][yi, :, :].values
    n_neg = (ems < 0).sum()
    assert n_neg == 0, f"EDGAR year {edgar_year_labels[yi]}: {n_neg} cells with negative emissions"

# no all-zero years — would indicate a bad/empty input file
for yi in range(num_years):
    total = float(EDGAR_aggregated_flux['emissions'][yi, :, :].sum().values)
    assert total > 0, f"EDGAR year {edgar_year_labels[yi]}: all-zero emissions (bad input file?)"

# conservation check: 0.1deg -> 1deg aggregation should preserve global totals
for yi in range(num_years):
    orig_total = float(EDGAR_flux['emissions'][yi, :, :].pint.to('Mg').sum().values)
    agg_total = float(EDGAR_aggregated_flux['emissions'][yi, :, :].sum().values)
    rel_err = abs(orig_total - agg_total) / max(abs(orig_total), 1e-30)
    assert rel_err < 1e-6, f"EDGAR year {yi}: aggregation changed total by {rel_err:.2e} (0.1deg: {orig_total:.4e}, 1deg: {agg_total:.4e})"

# normalized patterns should sum to 1.0 for each year
for yi in range(num_years):
    norm_sum = float(EDGAR_aggregated_flux['normalized_mass_tendency'][yi, :, :].sum().values)
    assert abs(norm_sum - 1.0) < 1e-10, f"EDGAR year {yi}: normalized_mass_tendency sums to {norm_sum}, expected 1.0"

print(f"  EDGAR checks OK: {num_years} years [{edgar_year_labels[0]}-{edgar_year_labels[-1]}], non-negative, no all-zero, conservation < 1e-6, normalization = 1.0")

EDGAR_aggregated_flux.pint.dequantify().to_netcdf('processed_inputs/EDGAR_fluxes.nc')
display(EDGAR_aggregated_flux)
EDGAR_aggregated_flux['flux'][0,:,:].plot(robust=True)

#### Dump EDGAR to IDL

In [ ]:
import os, sys
current_directory = os.getcwd()

# NOTE: this import fucks with your working directory. Use caution.
correct_dir = os.path.abspath('')
from idlpy import * 
# sys.path.append('/usr/local/nv5/idl/lib/bridges')
# sys.path.append('/Applications/NV5/idl91/lib/bridges')
os.chdir(correct_dir)

In [ ]:
edgar_array = EDGAR_aggregated_flux['normalized_mass_tendency'].to_numpy().transpose((1,2,0))

IDL.fracarr = edgar_array # do this funky way to specify variable name
IDL.run(command="SAVE, fracarr, FILENAME='processed_inputs/fracarr_2026.sav'")

### Load countries gridded map

In [ ]:
map = np.loadtxt('inputs/COUNTRY1X1.1993.mod.txt', skiprows=3)
map = map.reshape(180, 360)
map = xr.DataArray(map)

codes = pd.read_csv('inputs/COUNTRY1X1.CODE.mod2.2013.csv', header=None, names=['code', 'nation'])

# CDIAC country list and codes file must be the same length (positional alignment)
assert len(CDIAC_countries) == len(codes), f"CDIAC countries ({len(CDIAC_countries)}) != codes file ({len(codes)}) -- positional mapping will be wrong"

# every CDIAC country should have its code (or sub-region codes) present in the GISS map
codes_in_map = set(np.unique(map.values).astype(int)) - {0}  # 0 = ocean/unassigned

missing_from_map = []
for i in range(len(CDIAC_countries)):
    code = codes.iloc[i]['code']
    # large countries are subdivided: code 800 -> 801,802,...; check base code or sub-codes
    has_base = code in codes_in_map
    has_sub = any(c // 100 == code // 100 and c % 100 != 0 for c in codes_in_map) if not has_base else False
    if not has_base and not has_sub:
        missing_from_map.append((CDIAC_countries.iloc[i], code))

if missing_from_map:
    print(f"  WARNING: {len(missing_from_map)} CDIAC countries have no cells in GISS map: {missing_from_map[:10]}")
else:
    print(f"  Country-code coverage OK: all {len(CDIAC_countries)} CDIAC countries found in GISS map (including subdivided)")

map.plot()

In [ ]:
from sysconfig import get_path
print(get_path('purelib'))